# Predict — use the trained model without retraining

Loads the saved weights (`best_model.pt`) and applies the model to any original image. No training happens here.

The model definition and `DOWNSCALE` must match what was used during training (`model.ipynb`).

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

CKPT_PATH = os.path.join(os.getcwd(), "best_model.pt")
DOWNSCALE = (144, 96)   # must match training
BASE_CH = 32            # must match training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| checkpoint exists:", os.path.exists(CKPT_PATH))

In [ ]:
# --- Model definition (identical to model.ipynb) ---
def gn(c):
    return nn.GroupNorm(num_groups=min(8, c), num_channels=c)


class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), gn(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), gn(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class ResUNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        b = base
        self.d1 = DoubleConv(1, b)
        self.d2 = DoubleConv(b, b * 2)
        self.d3 = DoubleConv(b * 2, b * 4)
        self.d4 = DoubleConv(b * 4, b * 8)
        self.pool = nn.MaxPool2d(2)
        self.bott = DoubleConv(b * 8, b * 16)
        self.up4 = nn.ConvTranspose2d(b * 16, b * 8, 2, stride=2)
        self.u4 = DoubleConv(b * 16, b * 8)
        self.up3 = nn.ConvTranspose2d(b * 8, b * 4, 2, stride=2)
        self.u3 = DoubleConv(b * 8, b * 4)
        self.up2 = nn.ConvTranspose2d(b * 4, b * 2, 2, stride=2)
        self.u2 = DoubleConv(b * 4, b * 2)
        self.up1 = nn.ConvTranspose2d(b * 2, b, 2, stride=2)
        self.u1 = DoubleConv(b * 2, b)
        self.out = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        inp = x
        h, w = x.shape[-2:]
        m = 16
        ph, pw = (m - h % m) % m, (m - w % m) % m
        if ph or pw:
            x = F.pad(x, (0, pw, 0, ph), mode="reflect")
        c1 = self.d1(x)
        c2 = self.d2(self.pool(c1))
        c3 = self.d3(self.pool(c2))
        c4 = self.d4(self.pool(c3))
        bn = self.bott(self.pool(c4))
        y = self.u4(torch.cat([self.up4(bn), c4], 1))
        y = self.u3(torch.cat([self.up3(y), c3], 1))
        y = self.u2(torch.cat([self.up2(y), c2], 1))
        y = self.u1(torch.cat([self.up1(y), c1], 1))
        residual = self.out(y)[..., :h, :w]
        return torch.clamp(inp + residual, 0.0, 1.0)

In [ ]:
# --- Load the trained weights ONCE ---
model = ResUNet(BASE_CH).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()
print("model loaded from", CKPT_PATH)

In [ ]:
@torch.no_grad()
def enhance(image_path, out_path=None, keep_original_size=True):
    """Load an original image, run the model, return the enhanced PIL image.
    If out_path is given, also saves it. If keep_original_size, the result is
    resized back to the input's original resolution for viewing."""
    im = Image.open(image_path).convert("L")
    orig_size = im.size  # (w, h)
    x = im.resize(DOWNSCALE, Image.LANCZOS)
    t = torch.from_numpy(np.asarray(x, np.float32) / 255.0)[None, None].to(DEVICE)
    pred = model(t).cpu()[0, 0].numpy()
    out = Image.fromarray((pred * 255.0).round().astype(np.uint8), mode="L")
    if keep_original_size:
        out = out.resize(orig_size, Image.LANCZOS)
    if out_path:
        out.save(out_path, quality=95)
        print("saved", out_path)
    return out

## Comparison metrics

Compare the model output against the ground-truth edited image. Metrics are computed at `DOWNSCALE` resolution (same as training eval): **relative-MAE** (your metric, lower = better), **PSNR** (dB, higher = better), **SSIM** (−1…1, higher = better). Each comparison also reports the **identity baseline** (original vs edited) so you can see how much the model improved.

In [ ]:
try:
    from skimage.metrics import structural_similarity
    HAS_SSIM = True
except Exception:
    HAS_SSIM = False
    print("scikit-image not installed; SSIM will be NaN.")


def metrics_np(pred, target):
    """pred, target: float grayscale arrays in [0,255]. Returns (relative_mae, psnr, ssim)."""
    pred = np.asarray(pred, np.float32)
    target = np.asarray(target, np.float32)
    rmae = float(np.sum(np.abs(pred - target)) / (np.sum(target) + 1e-8))
    mse = float(np.mean((pred - target) ** 2))
    psnr = float("inf") if mse == 0 else float(20 * np.log10(255.0) - 10 * np.log10(mse))
    ssim = structural_similarity(target, pred, data_range=255.0) if HAS_SSIM else float("nan")
    return rmae, psnr, ssim


def _load_gray_ds(path):
    """Load grayscale, resize to DOWNSCALE, return float array in [0,255]."""
    im = Image.open(path).convert("L").resize(DOWNSCALE, Image.LANCZOS)
    return np.asarray(im, np.float32)


@torch.no_grad()
def compare_pair(original_path, edited_path, verbose=True):
    """Run the model on the original and compare to the edited (ground truth).
    Returns dict with model metrics and identity-baseline metrics."""
    o = _load_gray_ds(original_path)
    e = _load_gray_ds(edited_path)
    t = torch.from_numpy(o / 255.0)[None, None].to(DEVICE)
    pred = model(t).cpu()[0, 0].numpy() * 255.0
    model_m = metrics_np(pred, e)
    base_m = metrics_np(o, e)
    if verbose:
        print(f"{'':10s} {'rel_MAE':>10s} {'PSNR':>10s} {'SSIM':>10s}")
        print(f"{'baseline':10s} {base_m[0]:10.4f} {base_m[1]:10.3f} {base_m[2]:10.4f}")
        print(f"{'model':10s} {model_m[0]:10.4f} {model_m[1]:10.3f} {model_m[2]:10.4f}")
    return {"model": model_m, "baseline": base_m}

In [ ]:
import csv


def compare_all(original_dir, edited_dir, csv_path=None, show_worst=5):
    """Compare every output in a folder against ground truth.
    Prints averages, per-metric win-rate (how often the model beats the baseline),
    and the worst images by model rel_MAE. Optionally saves per-image CSV.
    Returns a list of (filename, model_metrics, baseline_metrics)."""
    o = set(f for f in os.listdir(original_dir) if f.lower().endswith(".jpg"))
    e = set(f for f in os.listdir(edited_dir) if f.lower().endswith(".jpg"))
    common = sorted(o & e)

    rows = []
    accm = np.zeros(3); accb = np.zeros(3); wins = np.zeros(3)
    for f in common:
        r = compare_pair(os.path.join(original_dir, f), os.path.join(edited_dir, f), verbose=False)
        m, b = np.array(r["model"]), np.array(r["baseline"])
        rows.append((f, m, b))
        accm += m; accb += b
        wins += np.array([m[0] < b[0], m[1] > b[1], m[2] > b[2]])  # is model better?

    n = max(len(common), 1)
    am, ab = accm / n, accb / n
    print(f"compared {len(common)} images\n")
    print(f"{'':10s} {'rel_MAE':>10s} {'PSNR':>10s} {'SSIM':>10s}")
    print(f"{'baseline':10s} {ab[0]:10.4f} {ab[1]:10.3f} {ab[2]:10.4f}")
    print(f"{'model':10s} {am[0]:10.4f} {am[1]:10.3f} {am[2]:10.4f}")
    print(f"\nmodel beats baseline:  rel_MAE {int(wins[0])}/{n}   "
          f"PSNR {int(wins[1])}/{n}   SSIM {int(wins[2])}/{n}")

    worst = sorted(rows, key=lambda x: x[1][0], reverse=True)[:show_worst]
    print(f"\nworst {len(worst)} by model rel_MAE:")
    for f, m, b in worst:
        print(f"   {f:16s} model rel_MAE {m[0]:.4f}  (baseline {b[0]:.4f})")

    if csv_path:
        with open(csv_path, "w", newline="") as fp:
            w = csv.writer(fp)
            w.writerow(["file", "model_relMAE", "model_PSNR", "model_SSIM",
                        "base_relMAE", "base_PSNR", "base_SSIM"])
            for f, m, b in rows:
                w.writerow([f, *[f"{v:.6f}" for v in m], *[f"{v:.6f}" for v in b]])
        print("\nper-image metrics saved to", csv_path)
    return rows

In [ ]:
# --- Compare ALL test outputs at once ---
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
orig_dir = os.path.join(DATA_DIR, "720*480", "test", "original")
edit_dir = os.path.join(DATA_DIR, "720*480", "test", "edited")

rows = compare_all(orig_dir, edit_dir, csv_path=os.path.join(os.getcwd(), "test_metrics.csv"))

In [ ]:
# --- Single image: enhance, display, compare to its edited ground truth ---
import matplotlib.pyplot as plt

example = sorted(f for f in os.listdir(orig_dir) if f.lower().endswith(".jpg"))[0]
src = os.path.join(orig_dir, example)
tgt = os.path.join(edit_dir, example)

result = enhance(src)
print(f"comparison for {example}:")
compare_pair(src, tgt)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(Image.open(src).convert("L"), cmap="gray"); ax[0].set_title(f"original — {example}"); ax[0].axis("off")
ax[1].imshow(result, cmap="gray"); ax[1].set_title("model output"); ax[1].axis("off")
ax[2].imshow(Image.open(tgt).convert("L"), cmap="gray"); ax[2].set_title("edited (ground truth)"); ax[2].axis("off")
plt.tight_layout(); plt.show()

## Worst test images by rel_MAE

The `K` test images with the highest model rel_MAE (the hardest cases), shown as **original | edited | predicted**, all at 144×96 (`DOWNSCALE`) resolution.

In [ ]:
# --- Top-K worst test images by model rel_MAE (all at 144x96) ---
import matplotlib.pyplot as plt

K = 5  # how many of the worst images to show

scored = []
for f in sorted(os.listdir(orig_dir)):
    if not f.lower().endswith(".jpg"):
        continue
    o = _load_gray_ds(os.path.join(orig_dir, f))   # 144x96, [0,255]
    e = _load_gray_ds(os.path.join(edit_dir, f))
    t = torch.from_numpy(o / 255.0)[None, None].to(DEVICE)
    with torch.no_grad():
        pred = model(t).cpu()[0, 0].numpy() * 255.0  # 144x96, [0,255]
    rmae = metrics_np(pred, e)[0]
    scored.append((rmae, f, o, e, pred))

scored.sort(key=lambda x: x[0], reverse=True)   # highest rel_MAE first
worst = scored[:K]

fig, axes = plt.subplots(len(worst), 3, figsize=(9, 3 * len(worst)))
if len(worst) == 1:
    axes = axes[None, :]
for row, (rmae, f, o, e, pred) in enumerate(worst):
    panels = [(o, f"original - {f}"), (e, "edited"), (pred, f"predicted (rel_MAE {rmae:.3f})")]
    for col, (img, title) in enumerate(panels):
        axes[row, col].imshow(img, cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(title, fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


## Best test images by rel_MAE

The `K` test images with the **lowest** model rel_MAE (where the model matches the edited target most closely), shown as **original | edited | predicted**, all at 144×96 (`DOWNSCALE`) resolution.

In [ ]:
# --- Top-K best test images by model rel_MAE (all at 144x96) ---
import matplotlib.pyplot as plt

K = 5  # how many of the best images to show

# Reuse `scored` from the previous block if available; otherwise recompute.
if "scored" not in globals():
    scored = []
    for f in sorted(os.listdir(orig_dir)):
        if not f.lower().endswith(".jpg"):
            continue
        o = _load_gray_ds(os.path.join(orig_dir, f))
        e = _load_gray_ds(os.path.join(edit_dir, f))
        t = torch.from_numpy(o / 255.0)[None, None].to(DEVICE)
        with torch.no_grad():
            pred = model(t).cpu()[0, 0].numpy() * 255.0
        scored.append((metrics_np(pred, e)[0], f, o, e, pred))

best = sorted(scored, key=lambda x: x[0])[:K]   # lowest rel_MAE first

fig, axes = plt.subplots(len(best), 3, figsize=(9, 3 * len(best)))
if len(best) == 1:
    axes = axes[None, :]
for row, (rmae, f, o, e, pred) in enumerate(best):
    panels = [(o, f"original - {f}"), (e, "edited"), (pred, f"predicted (rel_MAE {rmae:.3f})")]
    for col, (img, title) in enumerate(panels):
        axes[row, col].imshow(img, cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(title, fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()
